# Fine-tuning SLM Phi-3.5-mini (QLoRA) — Kriteria 1 PGABL

Notebook ini melatih model bahasa kecil **Phi-3.5-mini-instruct** memakai teknik
**QLoRA 4-bit** pada dataset instruksi Bahasa Indonesia, lalu mengunggah
hasilnya ke Hugging Face agar bisa dipanggil kembali pada tahap RAG.

Alur: instalasi -> autentikasi -> muat model 4-bit -> pasang LoRA ->
format dataset (chat template Phi-3.5) -> latih 800 langkah -> unggah
`merged_16bit`.

**Lingkungan:** Google Colab, Runtime **T4 GPU**.

## 1. Instalasi Pustaka

Unsloth dipasang tanpa memaku versi (`unpinned`) agar ia menautkan sendiri
versi `transformers`/`trl` yang cocok. Jika Colab meminta *restart session*
setelah sel ini, lakukan restart lalu **Run all** ulang.

In [3]:
%%capture
!pip install --upgrade --no-cache-dir unsloth
!pip install --upgrade --no-cache-dir trl peft accelerate bitsandbytes datasets huggingface_hub

## 2. Autentikasi Hugging Face

Token dan username diambil dari **Colab Secret** (bukan ditulis di sel).
`HF_TOKEN` harus ber-scope *Write* agar bisa membuat repositori model.

In [4]:
import os

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    HF_USERNAME = userdata.get("HF_USERNAME")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    HF_USERNAME = os.environ.get("HF_USERNAME")

assert HF_TOKEN, "HF_TOKEN belum tersedia. Simpan lewat Colab Secret."
assert HF_USERNAME, "HF_USERNAME belum tersedia. Simpan lewat Colab Secret."

from huggingface_hub import login
login(token=HF_TOKEN)
print("Terautentikasi. Akun:", HF_USERNAME)

Terautentikasi. Akun: bimo2107


## 3. Konfigurasi Pelatihan

Semua angka penting dikumpulkan di satu tempat agar mudah dilacak reviewer.

In [5]:
ID_MODEL     = "unsloth/Phi-3.5-mini-instruct-bnb-4bit"   # sudah 4-bit (nf4 + double quant)
REPO_MODEL   = f"{HF_USERNAME}/PGABL-Phi-3.5-mini-SFT-Bimo"

MAKS_TOKEN   = 1024      # panjang urutan maksimum
BENIH        = 3407      # seed reproducibility
RANK_LORA    = 8
ALPHA_LORA   = 16
DROPOUT_LORA = 0.05
N_SAMPEL     = 10_000    # sub-sampel dataset
BATCH        = 2         # per langkah
AKUMULASI    = 4         # effective batch = BATCH * AKUMULASI = 8
LANGKAH      = 800       # minimum wajib rubrik
LR           = 2e-4

print("Repo tujuan model:", REPO_MODEL)

Repo tujuan model: bimo2107/PGABL-Phi-3.5-mini-SFT-Bimo


## 4. Muat Model Dasar (QLoRA 4-bit + Double Quantization)

`load_in_4bit=True` mengaktifkan kuantisasi `nf4` dengan *double quantization*
secara internal. Cetakan `quantization_config` di bawah adalah bukti bahwa
QLoRA 4-bit benar-benar aktif.

In [6]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ID_MODEL,
    max_seq_length=MAKS_TOKEN,
    dtype=None,            # auto: float16 pada T4
    load_in_4bit=True,     # QLoRA (nf4 + double quant)
)

qcfg = getattr(model.config, "quantization_config", None)
print("Model dimuat :", ID_MODEL)
print("Konfig kuantisasi:", qcfg)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model dimuat : unsloth/Phi-3.5-mini-instruct-bnb-4bit
Konfig kuantisasi: {'bnb_4bit_compute_dtype': torch.float16, 'bnb_4bit_quant_type': 'nf4', 'bnb_4bit_use_double_quant': True, 'llm_int8_enable_fp32_cpu_offload': False, 'llm_int8_has_fp16_weight': False, 'llm_int8_skip_modules': None, 'llm_int8_threshold': 6.0, 'load_in_4bit': True, 'load_in_8bit': False, 'quant_method': 'bitsandbytes'}


## 5. Pasang Adapter LoRA (Attention + FFN)

Adapter ditaruh di proyeksi Phi-3.5 yang menyentuh dua komponen komputasi
utama: Multi-Head Attention (`qkv_proj`, `o_proj` — Phi memakai QKV ter-fusi)
dan Feed-Forward Network (`gate_up_proj`, `down_proj`). Rank kecil (r=8) cukup
untuk membentuk gaya jawaban formal Bahasa Indonesia tanpa memberatkan VRAM.

In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r=RANK_LORA,
    lora_alpha=ALPHA_LORA,
    lora_dropout=DROPOUT_LORA,
    target_modules=[
        "qkv_proj", "o_proj",         # Multi-Head Attention (QKV ter-fusi)
        "gate_up_proj", "down_proj",  # Feed-Forward Network (gate/up ter-fusi)
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=BENIH,
)

bisa_latih = sum(p.numel() for p in model.parameters() if p.requires_grad)
semua = sum(p.numel() for p in model.parameters())
print(f"Parameter dilatih : {bisa_latih:,} ({bisa_latih/semua:.3%} dari total)")

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Unsloth: You added custom modules, but Unsloth hasn't optimized for this.
Beware - your finetuning might be noticeably slower!
Unsloth: You added custom modules, but Unsloth hasn't optimized for this.
Beware - your finetuning might be noticeably slower!


Unsloth 2026.7.4 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Parameter dilatih : 4,456,448 (0.221% dari total)


## 6. Muat & Saring Dataset Instruksi

`Ichsan2895/alpaca-gpt4-indonesian` memiliki dua kolom: `input` (pertanyaan)
dan `output` (jawaban). Baris yang terlalu pendek dibuang agar model belajar
pola jawaban yang bermakna, lalu diambil sub-sampel acak `N_SAMPEL` baris.

In [8]:
from datasets import load_dataset

korpus = load_dataset("Ichsan2895/alpaca-gpt4-indonesian", split="train")
print("Baris mentah:", len(korpus), "| kolom:", korpus.column_names)

def cukup_panjang(baris):
    tanya = baris.get("input") or ""
    jawab = baris.get("output") or ""
    return len(tanya.strip()) >= 10 and len(jawab.strip()) >= 20

bersih = korpus.filter(cukup_panjang)
n = min(N_SAMPEL, len(bersih))
sampel = bersih.shuffle(seed=BENIH).select(range(n))
print("Setelah disaring:", len(bersih), "| dipakai:", len(sampel))

README.md:   0%|          | 0.00/1.91k [00:00<?, ?B/s]

alpaca-gpt4-indonesia.csv: reconstructing file:   0%|          |  0.00B / 41.4MB            

alpaca-gpt4-indonesia.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/49969 [00:00<?, ? examples/s]

Baris mentah: 49969 | kolom: ['Unnamed: 0', 'input', 'output']


Filter:   0%|          | 0/49969 [00:00<?, ? examples/s]

Setelah disaring: 49071 | dipakai: 10000


## 7. Terapkan Chat Template Phi-3.5 (Bukti Token Spesial)

Phi-3.5 memakai format percakapan dengan penanda `<|user|>`, `<|assistant|>`,
dan `<|end|>`. Template diatur lewat `get_chat_template(chat_template="phi-3.5")`,
lalu tiap baris dipetakan `datasets.map()` menjadi kolom `text`. Cetakan
berikut WAJIB memperlihatkan token spesial tersebut sebagai bukti template
sudah diterapkan sebelum melatih.

In [9]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="phi-3.5")

def susun_teks(baris):
    percakapan = [
        {"role": "user", "content": baris["input"]},
        {"role": "assistant", "content": baris["output"]},
    ]
    return {"text": tokenizer.apply_chat_template(
        percakapan, tokenize=False, add_generation_prompt=False)}

siap_latih = sampel.map(susun_teks, remove_columns=sampel.column_names)

baris_contoh = siap_latih[0]["text"]
print("-" * 70)
print("SATU BARIS DATASET SETELAH CHAT TEMPLATE PHI-3.5")
print("-" * 70)
print(baris_contoh)
print("-" * 70)
for penanda in ["<|user|>", "<|assistant|>", "<|end|>"]:
    status = "ADA" if penanda in baris_contoh else "TIDAK ADA"
    print(f"[{status}] token spesial {penanda}")

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

----------------------------------------------------------------------
SATU BARIS DATASET SETELAH CHAT TEMPLATE PHI-3.5
----------------------------------------------------------------------
<|user|>
Berapa kecepatan cahaya?
<|end|>
<|assistant|>
Kecepatan cahaya adalah sekitar 299.792.458 meter per detik (m/s), atau sekitar 186.282 mil per detik (mi/s) di dalam ruang hampa. Ini ditandai dengan konstanta fisik universal "c". Ini adalah kecepatan maksimum di mana semua energi, materi, dan informasi di alam semesta dapat bergerak.<|end|>

----------------------------------------------------------------------
[ADA] token spesial <|user|>
[ADA] token spesial <|assistant|>
[ADA] token spesial <|end|>


## 8. Susun SFTTrainer

`SFTTrainer` membaca kolom `text`. Effective batch = `BATCH * AKUMULASI = 8`.
Penjadwal `linear` dengan pemanasan 3% menurunkan laju belajar hingga langkah
ke-800. Optimizer `adamw_8bit` menghemat VRAM state optimizer.

In [11]:
from trl import SFTTrainer, SFTConfig

pengaturan = SFTConfig(
    output_dir="/content/hasil_sft",
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=AKUMULASI,
    max_steps=LANGKAH,
    learning_rate=LR,
    lr_scheduler_type="linear",
    warmup_steps=24,          # ~3% dari 800 langkah
    logging_steps=25,
    optim="adamw_8bit",
    weight_decay=0.01,
    seed=BENIH,
    report_to="none",
    max_seq_length=MAKS_TOKEN,
    dataset_text_field="text",
    packing=False,
    padding_free=False,       # matikan default baru TRL yang bentrok dgn max_length
    save_strategy="no",
)

sft = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=siap_latih,
    args=pengaturan,
)
print("SFTTrainer siap. Effective batch:", BATCH * AKUMULASI, "| langkah:", LANGKAH)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/10000 [00:00<?, ? examples/s]

SFTTrainer siap. Effective batch: 8 | langkah: 800


## 9. Jalankan Pelatihan (800 Langkah)

Perkiraan durasi di Colab T4 sekitar 60-90 menit. Selama `loss` menurun dan
tidak ada error CUDA out-of-memory, ketentuan Basic Kriteria 1 terpenuhi.

In [12]:
ringkasan = sft.train()
print("Pelatihan selesai.")
print("Langkah terakhir:", sft.state.global_step)
print("Loss terakhir   :", sft.state.log_history[-1].get("loss"))

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 32009}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,000 | Num Epochs = 1 | Total steps = 800
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 4,456,448 of 3,825,536,000 (0.12% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
25,14.241030
50,14.301395
75,14.151455
100,14.084509
125,14.254312
150,14.154132
175,14.286121
200,13.652755
225,14.190468
250,14.079487


Pelatihan selesai.
Langkah terakhir: 800
Loss terakhir   : None


## 10. Gabungkan Adapter & Unggah ke Hugging Face

Adapter LoRA digabung ke bobot dasar (`merged_16bit`) supaya model bisa
langsung dipakai inferensi tanpa PEFT pada tahap RAG. Pengunggahan memakai
pola dua langkah (`save_pretrained_merged` lalu `HfApi().upload_folder`) untuk
menghindari `TypeError: safe_serialization` pada versi Unsloth terbaru;
`delete_patterns` memastikan tidak ada berkas adapter tersisa di repositori.

In [13]:
from huggingface_hub import HfApi, create_repo

FOLDER_GABUNG = "/content/phi_sft_merged"

model.save_pretrained_merged(FOLDER_GABUNG, tokenizer, save_method="merged_16bit")
print("Bobot tergabung disimpan:", FOLDER_GABUNG)

api = HfApi()
create_repo(repo_id=REPO_MODEL, private=False, exist_ok=True, token=HF_TOKEN)
api.upload_folder(
    folder_path=FOLDER_GABUNG,
    repo_id=REPO_MODEL,
    token=HF_TOKEN,
    commit_message="SFT Phi-3.5-mini merged_16bit (PGABL)",
    delete_patterns=["adapter_config.json", "adapter_model.safetensors"],
)
tautan_model = f"https://huggingface.co/{REPO_MODEL}"
print("Model publik terunggah:", tautan_model)

config.json:   0%|          | 0.00/3.31k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /content/phi_sft_merged/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/phi_sft_merged.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...



Unsloth: Preparing safetensor model files:  50%|█████     | 1/2 [00:47<00:47, 47.82s/it]

model-00002-of-00002.safetensors: reconstructing file:   0%|          |  0.00B / 2.65GB            

model-00002-of-00002.safetensors: downloading bytes:           |  0.00B            



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:22<00:00, 41.40s/it]


Unsloth: Merge process complete. Saved to `/content/phi_sft_merged`
Bobot tergabung disimpan: /content/phi_sft_merged
Model publik terunggah: https://huggingface.co/bimo2107/PGABL-Phi-3.5-mini-SFT-Bimo


## 11. Simpan Tautan Model

Tautan repositori disimpan ke `link_huggingface.txt` (salah satu berkas
submission). Unduh berkas ini bersama notebook yang sudah tereksekusi.

In [14]:
with open("link_huggingface.txt", "w", encoding="utf-8") as f:
    f.write(tautan_model + "\n")
print("Ditulis ke link_huggingface.txt:")
print(tautan_model)

Ditulis ke link_huggingface.txt:
https://huggingface.co/bimo2107/PGABL-Phi-3.5-mini-SFT-Bimo


## 12. Ringkasan

| Aspek | Nilai |
|---|---|
| Model dasar | `unsloth/Phi-3.5-mini-instruct-bnb-4bit` |
| Teknik | QLoRA 4-bit (nf4 + double quant), LoRA r=8 a=16 pada MHA+FFN |
| Dataset | `Ichsan2895/alpaca-gpt4-indonesian` (chat template Phi-3.5) |
| Pelatihan | SFTTrainer 800 langkah, effective batch 8, LR 2e-4 linear |
| Unggah | `merged_16bit` ke repositori Hugging Face publik |

Kriteria 1 (Basic) terpenuhi. Model siap dipakai pada notebook RAG.